EDA (Exploratory Data Analysis)
This is the most underrated but most important skill in data work. Every data analyst need to this before writing a single pipeline.

In [32]:
import pandas as pd
import numpy as np
import os

# Load all files
RAW = '../data/raw/'

orders    = pd.read_csv(RAW + 'olist_orders_dataset.csv')
customers = pd.read_csv(RAW + 'olist_customers_dataset.csv')
items     = pd.read_csv(RAW + 'olist_order_items_dataset.csv')
products  = pd.read_csv(RAW + 'olist_products_dataset.csv')
sellers   = pd.read_csv(RAW + 'olist_sellers_dataset.csv')
payments  = pd.read_csv(RAW + 'olist_order_payments_dataset.csv')
reviews   = pd.read_csv(RAW + 'olist_order_reviews_dataset.csv')

print("✅ All files loaded")

✅ All files loaded


In [33]:
print("SHAPE:", orders.shape)
print("\nCOLUMNS & DATATYPES:")
print(orders.dtypes)
print("\nFIRST 3 ROWS:")
orders.head(3)

SHAPE: (99441, 8)

COLUMNS & DATATYPES:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

FIRST 3 ROWS:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [34]:
# How many orders per status?
print(orders['order_status'].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [35]:
# How many nulls in each column?
print("\nNULL COUNTS:")
print(orders.isnull().sum())


NULL COUNTS:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [36]:
print("SHAPE:", items.shape)
print("\nCOLUMNS:")
print(items.dtypes)
print("\nSTATISTICAL SUMMARY:")
items[['price', 'freight_value']].describe()


SHAPE: (112650, 7)

COLUMNS:
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

STATISTICAL SUMMARY:


,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


price = product price per item
freight_value = shipping cost
One order can have multiple items — that's why items has more rows than orders
price + freight_value = total revenue per line item — we'll compute this in transform

In [37]:
# Verify: orders with multiple items
items.groupby('order_id')['order_item_id'].max().value_counts().head()

order_item_id
1    88863
2     7516
3     1322
4      505
5      204
Name: count, dtype: int64

In [38]:
# Customers
print("CUSTOMERS:")
print(customers.dtypes)
print("\nUnique states:", customers['customer_state'].nunique())
print(customers['customer_state'].value_counts().head(5))

print("\n" + "="*40)

# Sellers
print("\nSELLERS:")
print(sellers.dtypes)
print("\nUnique seller states:", sellers['seller_state'].nunique())
print(sellers['seller_state'].value_counts().head(5))

CUSTOMERS:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

Unique states: 27
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
Name: count, dtype: int64


SELLERS:
seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object

Unique seller states: 23
seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
Name: count, dtype: int64


In [39]:
print("PAYMENT TYPES:")
print(payments['payment_type'].value_counts())

print("\nPAYMENT VALUE STATS:")
print(payments['payment_value'].describe())

# Average installments per payment type
print("\nAVG INSTALLMENTS BY TYPE:")
print(payments.groupby('payment_type')['payment_installments'].mean().round(1))


PAYMENT TYPES:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

PAYMENT VALUE STATS:
count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64

AVG INSTALLMENTS BY TYPE:
payment_type
boleto         1.0
credit_card    3.5
debit_card     1.0
not_defined    1.0
voucher        1.0
Name: payment_installments, dtype: float64


In [40]:
print("REVIEW SCORE DISTRIBUTION:")
print(reviews['review_score'].value_counts().sort_index())

# What % of reviews are positive (4 or 5)?
positive = reviews[reviews['review_score'] >= 4].shape[0]
total = reviews.shape[0]
print(f"\n✅ Positive reviews: {positive/total*100:.1f}%")

REVIEW SCORE DISTRIBUTION:
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

✅ Positive reviews: 77.1%


In [41]:
print("PRODUCT CATEGORIES (Top 10):")
print(products['product_category_name'].value_counts().head(10))

print(f"\nTotal unique categories: {products['product_category_name'].nunique()}")
print(f"Products with missing category: {products['product_category_name'].isnull().sum()}")

PRODUCT CATEGORIES (Top 10):
product_category_name
cama_mesa_banho           3029
esporte_lazer             2867
moveis_decoracao          2657
beleza_saude              2444
utilidades_domesticas     2335
automotivo                1900
informatica_acessorios    1639
brinquedos                1411
relogios_presentes        1329
telefonia                 1134
Name: count, dtype: int64

Total unique categories: 73
Products with missing category: 610


In [42]:
# This is understanding your Star Schema before building it

print("🔗 KEY RELATIONSHIPS:")
print(f"\norders.customer_id   → customers.customer_id")
print(f"  Orders unique customers:  {orders['customer_id'].nunique():,}")
print(f"  Customers table rows:     {customers.shape[0]:,}")

print(f"\nitems.order_id       → orders.order_id")
print(f"  Unique orders in items:   {items['order_id'].nunique():,}")
print(f"  Total orders:             {orders['order_id'].nunique():,}")

print(f"\nitems.product_id     → products.product_id")
print(f"  Unique products in items: {items['product_id'].nunique():,}")
print(f"  Total products:           {products['product_id'].nunique():,}")

print(f"\nitems.seller_id      → sellers.seller_id")
print(f"  Unique sellers in items:  {items['seller_id'].nunique():,}")
print(f"  Total sellers:            {sellers['seller_id'].nunique():,}")

🔗 KEY RELATIONSHIPS:

orders.customer_id   → customers.customer_id
  Orders unique customers:  99,441
  Customers table rows:     99,441

items.order_id       → orders.order_id
  Unique orders in items:   98,666
  Total orders:             99,441

items.product_id     → products.product_id
  Unique products in items: 32,951
  Total products:           32,951

items.seller_id      → sellers.seller_id
  Unique sellers in items:  3,095
  Total sellers:            3,095


In [43]:


eda_notes = {
    "orders_nulls"       : "delivery dates have nulls for non-delivered orders",
    "date_columns_issue" : "all dates stored as strings — need pd.to_datetime()",
    "revenue_location"   : "price and freight_value are in ITEMS, not orders",
    "focus_status"       : "filter to 'delivered' orders for analysis",
    "product_nulls"      : "some products have missing category name",
    "dominant_payment"   : "credit_card is most common payment type",
    "review_skew"        : "majority of reviews are 5-star (positive skew)",
}

print("📋 MY EDA NOTES:")
for k, v in eda_notes.items():
    print(f"  {k:<25} → {v}")

📋 MY EDA NOTES:
  orders_nulls              → delivery dates have nulls for non-delivered orders
  date_columns_issue        → all dates stored as strings — need pd.to_datetime()
  revenue_location          → price and freight_value are in ITEMS, not orders
  focus_status              → filter to 'delivered' orders for analysis
  product_nulls             → some products have missing category name
  dominant_payment          → credit_card is most common payment type
  review_skew               → majority of reviews are 5-star (positive skew)


In [44]:
# print info of all tables
print(items.info())
print(customers.info())
print(orders.info())
print(reviews.info())
print(sellers.info())
print(payments.info())
print(products.info())


<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB
None
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_cit